In [ ]:

import sys
import os
import math
import traceback

import torch
import torch.nn as nn
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


ROOT = os.path.abspath("..")          # one level up from notebooks/
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import config
from modules.m2_embeddings import TokenEmbedding, RotaryEmbedding


os.makedirs(os.path.join(ROOT, "results"), exist_ok=True)
RESULTS = os.path.join(ROOT, "results")


VOCAB     = config.VOCAB_SIZE    # 32 000
DIM       = config.D_MODEL     # 256
MLEN      = config.MAX_SEQ_LEN   # 256
HEAD_DIM  = DIM // config.N_HEADS  # 32  (256 / 8)


def passed(msg: str) -> None:
    print(f"  ✅  PASS  {msg}")

def failed(msg: str) -> None:
    print(f"  ❌  FAIL  {msg}")

def result(condition: bool, msg_pass: str, msg_fail: str = "") -> None:
    """Print PASS or FAIL based on condition."""
    if condition:
        passed(msg_pass)
    else:
        failed(msg_fail or msg_pass)

def section(title: str) -> None:
    bar = "═" * 60
    print(f"\n{bar}\n  {title}\n{bar}")

def sub(title: str) -> None:
    print(f"\n  ── {title}")

# ── Verify the environment is sane ────────────────────────────────────────────
section("CELL 1: SETUP")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  PyTorch     : {torch.__version__}")
print(f"  CUDA avail  : {torch.cuda.is_available()}")
print(f"  config.VOCAB_SIZE   = {VOCAB:,}")
print(f"  config.EMBED_DIM    = {DIM}")
print(f"  config.MAX_SEQ_LEN  = {MLEN}")
print(f"  config.N_HEADS      = {config.N_HEADS}")
print(f"  HEAD_DIM            = {HEAD_DIM}")
print(f"  results dir : {RESULTS}")
passed("Setup complete — all cells below are ready to run")

In [ ]:
# ═══ TEST 1 — TokenEmbedding Output Shape ════════════════════════════
#
# WHAT:  Feed random token IDs through TokenEmbedding and verify shape.
# WHY:   Shape errors are the most common crash in production.
#        If the output shape is wrong, every downstream module breaks.
#
# VISUAL:
#   input_ids   [batch, seq_len]          = [16, 256]   integers
#       ↓  nn.Embedding
#   vectors     [batch, seq_len, dim]     = [16, 256, 256]  floats
#       ↓  RoPE + LayerNorm + Dropout
#   output      [batch, seq_len, dim]     = [16, 256, 256]  floats
#
# MATRICES INVOLVED:
#   Embedding weight : [32000, 256]  — the lookup table
#   Input            : [16, 256]    — integer indices
#   Output           : [16, 256, 256] — float vectors
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 1: TokenEmbedding Output Shape")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
emb.eval()   # disable dropout so output is deterministic

BATCH   = 16
SEQ     = MLEN  # 256

fake_ids = torch.randint(0, VOCAB, (BATCH, SEQ))   # [16, 256]
output   = emb(fake_ids)                           # should be [16, 256, 256]

EXPECTED = torch.Size([BATCH, SEQ, DIM])

print(f"\n  Input  shape : {fake_ids.shape}   dtype={fake_ids.dtype}")
print(f"  Output shape : {output.shape}  dtype={output.dtype}")
print(f"  Expected     : {EXPECTED}")

result(output.shape == EXPECTED,
       f"Shape correct {output.shape}",
       f"Shape WRONG — got {output.shape}, expected {EXPECTED}")

result(output.dtype == torch.float32,
       "Output dtype is float32",
       f"Output dtype is {output.dtype} — expected float32")

result(not torch.isnan(output).any(),
       "No NaN values in output",
       "NaN values found in output!")

result(not torch.isinf(output).any(),
       "No Inf values in output",
       "Inf values found in output!")

# ── VISUAL: show a slice ──────────────────────────────────────────────────────
print(f"\n  First token of first batch item (first 8 dims):")
print(f"  {output[0, 0, :8].tolist()}")
print(f"\n  These numbers have no meaning yet — they are the raw embeddings")
print(f"  for token ID {fake_ids[0,0].item()} before the model has trained.")

In [ ]:
# ═══ TEST 2 — RoPE Position Sensitivity ══════════════════════════════
#
# WHAT:  Verify that the SAME token vector gets DIFFERENT output at
#        different sequence positions.
# WHY:   This is the ENTIRE POINT of RoPE. If this fails, the model
#        has no position information — all positions look identical.
#
# INTUITION:
#   Imagine a compass needle. Its value depends on which direction it points.
#   RoPE "points" each token vector in a direction determined by its position.
#   Token at pos 0 points North. Token at pos 1 points NNE. Etc.
#   Same word at pos 0 vs pos 5 → very different vectors.
#
# HOW WE TEST:
#   1. Make one random vector
#   2. Copy it to every position in a sequence (positions 0..9)
#   3. Run RoPE
#   4. Check that output at position 0 ≠ output at position 5
#   5. Check that distance INCREASES with position gap
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 2: RoPE Position Sensitivity")

rope = RotaryEmbedding(dim=DIM, max_len=MLEN)

# One random vector, copied to 10 positions
# Shape: [1 batch, 10 positions, dim]
base_vec = torch.randn(1, 1, DIM)
seq = base_vec.expand(1, 10, DIM).clone()  # [1, 10, DIM]

print(f"\n  Input (same vector at all 10 positions):")
print(f"  Position 0: {seq[0, 0, :4].tolist()}")
print(f"  Position 5: {seq[0, 5, :4].tolist()}")
print(f"  Equal before RoPE: {torch.allclose(seq[0,0], seq[0,5])}")

rotated = rope(seq)  # [1, 10, DIM]

print(f"\n  After RoPE:")
print(f"  Position 0: {rotated[0, 0, :4].tolist()}")
print(f"  Position 5: {rotated[0, 5, :4].tolist()}")

are_different = not torch.allclose(rotated[0, 0], rotated[0, 5], atol=1e-6)
result(are_different,
       "Position 0 ≠ Position 5 (position IS encoded)",
       "Position 0 == Position 5 (position NOT encoded — bug!)")

# ── Position 0 should be IDENTITY (rotation by 0 radians) ────────────────────
# cos(0)=1, sin(0)=0 → new_even = even*1 - odd*0 = even → unchanged
sub("Position 0 identity check (rotation by 0° leaves vector unchanged)")
pos0_unchanged = torch.allclose(seq[0, 0], rotated[0, 0], atol=1e-6)
result(pos0_unchanged,
       "Position 0 output == input (correct: angle=0 means no rotation)",
       "Position 0 output ≠ input (bug: zero angle should not rotate!)")

# ── Distance should INCREASE with position gap ────────────────────────────────
sub("Distance grows with position gap")
print(f"\n  Distances from position 0:")
prev_dist = 0.0
monotone  = True
for pos in [1, 2, 3, 5, 9]:
    d = (rotated[0, 0] - rotated[0, pos]).norm().item()
    print(f"    pos 0 vs pos {pos:2d} : distance = {d:.4f}")
    if d < prev_dist - 1e-4:          # allow tiny floating-point noise
        monotone = False
    prev_dist = d

result(monotone,
       "Distance increases monotonically with position gap ✓",
       "Distance does NOT increase monotonically — something is wrong")

In [ ]:
# ═══ TEST 3 — RoPE Shape Preservation ════════════════════════════════
#
# WHAT:  RoPE must return the EXACT same shape it receives.
#        Test many shapes including sizes LARGER than training max.
# WHY:   In production, inference sequences are often longer than
#        the training maximum. RoPE should still work because we
#        precomputed the table for 4× max_len.
#
# SHAPES TESTED:
#   [1,  1,   1] — single token, minimum possible
#   [1,  256, DIM] — full training size
#   [16, 256, DIM] — full batch
#   [1,  512, DIM] — 2× training length (generalization test)
#   [1,  1024,DIM] — 4× training length (at our table limit)
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 3: RoPE Shape Preservation (including out-of-training lengths)")

rope = RotaryEmbedding(dim=DIM, max_len=MLEN * 4)  # table covers 4× training

shapes_to_test = [
    (1,   1,        DIM,  "single token"),
    (1,   10,       DIM,  "short sequence"),
    (1,   MLEN,     DIM,  "full training length"),
    (16,  MLEN,     DIM,  "full batch, full length"),
    (1,   MLEN * 2, DIM,  "2× training length (generalization)"),
    (1,   MLEN * 4, DIM,  "4× training length (table limit)"),
]

for batch, seq, dim, label in shapes_to_test:
    x = torch.randn(batch, seq, dim)
    try:
        y = rope(x)
        shape_ok = (y.shape == x.shape)
        nan_ok   = not torch.isnan(y).any()
        inf_ok   = not torch.isinf(y).any()
        all_ok   = shape_ok and nan_ok and inf_ok

        status   = "✅" if all_ok else "❌"
        issues   = []
        if not shape_ok: issues.append(f"shape {y.shape}≠{x.shape}")
        if not nan_ok:   issues.append("NaN")
        if not inf_ok:   issues.append("Inf")
        issue_str = ", ".join(issues) if issues else ""

        print(f"  {status}  [{batch:2d}, {seq:4d}, {dim}]  {label}  {issue_str}")
    except Exception as e:
        print(f"  ❌  [{batch:2d}, {seq:4d}, {dim}]  {label}  EXCEPTION: {e}")

# ── What happens BEYOND the table? ───────────────────────────────────────────
sub("Beyond table limit (5× training length) — should fail gracefully")
too_long = torch.randn(1, MLEN * 5, DIM)
try:
    y = rope(too_long)
    print(f"  ⚠️  Sequence beyond table limit DID NOT raise error.")
    print(f"      PyTorch will silently repeat or wrap angles — potential bug!")
except Exception as e:
    print(f"  ✅  Raises {type(e).__name__}: {e}")
    print(f"      In production: catch this and truncate or re-build the table.")

In [ ]:
# ═══ TEST 4 — Numerical Stability (NaN / Inf Guard) ══════════════════
#
# WHAT:  Feed extreme values and verify the output stays finite.
# WHY:   Real data is messy. Embeddings can occasionally blow up
#        (especially early in training with large learning rates).
#        We need to know exactly where the module breaks down.
#
# CASES:
#   - Normal random input         → baseline
#   - Very large values (×1000)   → embedding outputs might be huge
#   - Very small values (÷1000)   → underflow risk
#   - All zeros                   → degenerate but valid
#   - Mixed large & small         → realistic noisy training scenario
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 4: Numerical Stability — NaN / Inf / Extreme Values")

rope = RotaryEmbedding(dim=DIM, max_len=MLEN)

cases = [
    ("Normal random",        torch.randn(2, 16, DIM)),
    ("Large values ×1000",   torch.randn(2, 16, DIM) * 1000),
    ("Small values ÷1000",   torch.randn(2, 16, DIM) / 1000),
    ("All zeros",            torch.zeros(2, 16, DIM)),
    ("All ones",             torch.ones(2, 16, DIM)),
    ("Very large (1e6)",     torch.randn(2, 16, DIM) * 1e6),
    ("Very small (1e-6)",    torch.randn(2, 16, DIM) * 1e-6),
    ("Mixed: 50% large",     torch.cat([
                                 torch.randn(2, 8, DIM) * 1000,
                                 torch.randn(2, 8, DIM)
                             ], dim=1)),
]

print(f"\n  {'Case':<25}  {'NaN':>6}  {'Inf':>6}  {'Max':>12}  {'Status'}")
print(f"  {'─'*25}  {'─'*6}  {'─'*6}  {'─'*12}  {'─'*6}")

for label, x in cases:
    try:
        y         = rope(x)
        has_nan   = torch.isnan(y).any().item()
        has_inf   = torch.isinf(y).any().item()
        max_val   = y.abs().max().item()
        ok_flag   = not has_nan and not has_inf
        status    = "✅" if ok_flag else "❌"
        print(f"  {label:<25}  {str(has_nan):>6}  {str(has_inf):>6}  {max_val:>12.2f}  {status}")
    except Exception as e:
        print(f"  {label:<25}  EXCEPTION: {e}")

# ── Why RoPE is stable ────────────────────────────────────────────────────────
print("""
  WHY RoPE IS NUMERICALLY STABLE:
    RoPE only multiplies input values by cos/sin (both bounded in [-1, 1]).
    It never adds unbounded values, so large inputs stay large but not
    larger — output magnitude = input magnitude × |rotation| ≈ input.
    If input has NaN or Inf, output inherits it (GIGO — garbage in, garbage out).
""")

In [ ]:
# ═══  TEST 5 — Production Hazard: Wrong dtype ═════════════════════════
#
# WHAT:  What happens when the caller passes the wrong dtype?
# WHY:   In production, data pipelines mix dtypes.
#        - int64 IDs passed to RoPE (should go through Embedding first)
#        - float16 from mixed-precision training
#        - float64 from numpy conversion
#
# EXPECTED BEHAVIOUR:
#   - float32 : works perfectly (our native type)
#   - float16 : may work (PyTorch handles it) or may lose precision
#   - float64 : should work (more precision than needed)
#   - int64   : MUST raise — RoPE does arithmetic on floats, not ints
#   - int IDs to TokenEmbedding : must be int64 (LongTensor)
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 5: Production Hazard — Wrong Input dtype")

rope = RotaryEmbedding(dim=DIM, max_len=MLEN)
emb  = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)

sub("RoPE dtype handling")

dtype_cases = [
    (torch.float32, "float32 (correct)"),
    (torch.float16, "float16 (half precision — mixed training)"),
    (torch.float64, "float64 (double — from numpy)"),
]

for dtype, label in dtype_cases:
    x = torch.randn(1, 4, DIM, dtype=dtype)
    try:
        y = rope(x)
        print(f"  ✅  {label} → output dtype: {y.dtype}")
    except Exception as e:
        print(f"  ❌  {label} → {type(e).__name__}: {e}")

# Integer input to RoPE — should fail (wrong usage)
try:
    x_int = torch.randint(0, 100, (1, 4, DIM))  # int64
    y = rope(x_int)
    print(f"  ⚠️  int64 to RoPE worked (but results may be wrong — {y.dtype})")
except Exception as e:
    print(f"  ✅  int64 to RoPE correctly raises {type(e).__name__}: {e}")

sub("TokenEmbedding: token IDs dtype")
# Correct: int64 (LongTensor)
ids_long  = torch.randint(0, VOCAB, (1, 4), dtype=torch.long)
try:
    out = emb(ids_long)
    print(f"  ✅  int64 IDs → output shape {out.shape}")
except Exception as e:
    print(f"  ❌  int64 IDs failed: {e}")

# Wrong: float32 IDs
ids_float = torch.rand(1, 4) * VOCAB
try:
    out = emb(ids_float)
    print(f"  ⚠️  float32 IDs worked — PyTorch auto-converted (risky!)")
except Exception as e:
    print(f"  ✅  float32 IDs correctly raises {type(e).__name__}: {e}")
    print(f"      Fix: cast with ids.long() before passing to embedding")

# Wrong: int32 instead of int64
ids_int32 = torch.randint(0, VOCAB, (1, 4), dtype=torch.int32)
try:
    out = emb(ids_int32)
    print(f"  ⚠️  int32 IDs worked — PyTorch auto-converted")
except Exception as e:
    print(f"  ✅  int32 IDs raises {type(e).__name__}: {e}")
    print(f"      Fix: ids = ids.to(torch.long)")

In [ ]:
# ═══  TEST 6 — Production Hazard: Out-of-Vocabulary Token IDs ═════════
#
# WHAT:  What happens when a token ID >= vocab_size or < 0?
# WHY:   In production:
#        - New rare words may get an OOV (out-of-vocabulary) ID
#        - Tokenizer bugs can produce IDs beyond the table size
#        - Negative IDs can appear from buggy preprocessing
#
# EXPECTED:
#   - ID = vocab_size     → IndexError (one past the end)
#   - ID = vocab_size + 1 → IndexError
#   - ID = -1             → PyTorch wraps around (last row) — dangerous!
#   - ID = 0              → padding, gives zero vector (intentional)
#
# PRODUCTION FIX:
#   Always clamp IDs: ids = ids.clamp(0, vocab_size - 1)
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 6: Production Hazard — Out-of-Vocabulary IDs")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)

cases = [
    (VOCAB - 1,  "Last valid ID (vocab_size - 1)"),
    (VOCAB,      "One past end (vocab_size) — out of bounds"),
    (VOCAB + 99, "Far out of bounds"),
    (0,          "Padding ID (0) — should give zero vector"),
    (-1,         "Negative ID — wraps to last row in PyTorch!"),
]

for token_id, label in cases:
    ids = torch.tensor([[token_id]], dtype=torch.long)
    try:
        out = emb(ids)
        norm = out.norm().item()
        print(f"  ⚠️  ID={token_id:7d}  {label}")
        print(f"          Output norm={norm:.4f} (no error raised — may be silently wrong)")
    except IndexError as e:
        print(f"  ✅  ID={token_id:7d}  {label}")
        print(f"          Raises IndexError — correct, out-of-bounds detected")
    except Exception as e:
        print(f"  ⚠️  ID={token_id:7d}  {label}")
        print(f"          Raises {type(e).__name__}: {e}")

# ── Show the production fix ───────────────────────────────────────────────────
print("""
  PRODUCTION FIX:
    def safe_forward(emb_module, ids):
        ids = ids.clamp(min=0, max=emb_module.vocab_size - 1)
        return emb_module(ids)
""")

In [ ]:
# ═══  TEST 7 — Production Hazard: Device Mismatch ═════════════════════
#
# WHAT:  Model is on GPU, input is on CPU (or vice versa).
# WHY:   Most common PyTorch error in production.
#        Happens when you forget to call .to(device) on your inputs.
#
# EXPECTED:
#   - Model on CPU, input on CPU  → works (normal development)
#   - Model on CUDA, input on CPU → RuntimeError (must match)
#   - Model on CPU, input on CUDA → RuntimeError
#
# PRODUCTION FIX:
#   Always do: input_ids = input_ids.to(next(model.parameters()).device)
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 7: Production Hazard — Device Mismatch")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
fake_ids = torch.randint(0, VOCAB, (2, 10))

sub("CPU model, CPU input (normal case)")
try:
    out = emb(fake_ids)
    print(f"  ✅  CPU × CPU works — output on {out.device}")
except Exception as e:
    print(f"  ❌  {e}")

if torch.cuda.is_available():
    sub("CUDA model, CPU input (production hazard)")
    emb_cuda = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN).cuda()
    try:
        out = emb_cuda(fake_ids)       # input still on CPU!
        print(f"  ⚠️  Did not raise — may be silently wrong")
    except RuntimeError as e:
        print(f"  ✅  Raises RuntimeError: {str(e)[:80]}")
        print(f"      Fix: fake_ids = fake_ids.to('cuda')")

    sub("CUDA model, CUDA input (correct)")
    try:
        out = emb_cuda(fake_ids.cuda())
        print(f"  ✅  CUDA × CUDA works — output on {out.device}")
    except Exception as e:
        print(f"  ❌  {e}")
else:
    print("\n  (CUDA not available — skipping GPU tests)")
    print("  In production on GPU servers, run these checks!")

# ── Show the safe device pattern ─────────────────────────────────────────────
print("""
  PRODUCTION PATTERN:
    device   = next(model.parameters()).device
    ids      = ids.to(device)
    output   = model(ids)
""")

In [ ]:
# ═══  TEST 8 — Gradient Flow (Can the Model Learn?) ═══════════════════
#
# WHAT:  Verify gradients flow back through the embedding module.
# WHY:   If gradients are zero or None, the embedding table NEVER UPDATES.
#        The model would train (no error) but embeddings stay random forever.
#
# HOW:
#   1. Forward pass
#   2. Compute a fake loss (sum of all outputs)
#   3. Backward pass
#   4. Check that embedding.weight.grad is not None and not all-zero
#
# WHAT WE CHECK:
#   - embedding.weight.grad is not None
#   - embedding.weight.grad has no NaN
#   - At least some gradients are non-zero
#   - Gradient sparsity: only rows corresponding to seen tokens get gradient
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 8: Gradient Flow — Can the Model Actually Learn?")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
emb.train()   # enable dropout, enable grad

# Use specific token IDs so we can check which rows get gradients
token_ids = torch.tensor([[27, 342, 891, 27]])   # "I love Cats I"
# token 27 appears TWICE — it should get twice the gradient

output = emb(token_ids)
loss   = output.sum()            # simple scalar loss
loss.backward()

sub("Embedding weight gradient")
grad = emb.embedding.weight.grad

result(grad is not None,
       "Gradient exists on embedding.weight",
       "Gradient is None — backprop not reaching embedding!")

if grad is not None:
    nan_in_grad  = torch.isnan(grad).any().item()
    zero_grad    = (grad == 0).all().item()
    nonzero_rows = (grad.abs().sum(dim=1) > 0).sum().item()

    result(not nan_in_grad,
           "No NaN in gradients",
           "NaN in gradients — training would corrupt weights!")

    result(not zero_grad,
           "Gradients are not all zero",
           "All gradients are zero — model cannot learn!")

    print(f"\n  Gradient sparsity (nn.Embedding is sparse by design):")
    print(f"    Rows with non-zero gradient : {nonzero_rows} / {VOCAB}")
    print(f"    Expected: only the {len(set([27, 342, 891]))} unique tokens used in forward pass")
    print(f"    (nn.Embedding only computes gradients for rows it actually used)")

    # Verify token 27 has a gradient, token 0 (unseen) does not
    tok27_has_grad  = grad[27].abs().sum().item() > 0
    tok0_has_grad   = grad[0].abs().sum().item() > 0    # padding, not used

    result(tok27_has_grad,
           f"Token 27 ('I') has gradient — correct",
           f"Token 27 has zero gradient — should have one!")

    result(not tok0_has_grad,
           f"Token 0 (padding) has zero gradient — correct (padding_idx=0)",
           f"Token 0 has gradient — should not (padding_idx=0 prevents this)")

sub("LayerNorm gradient")
ln_grad = emb.norm.weight.grad
result(ln_grad is not None and ln_grad.abs().sum().item() > 0,
       "LayerNorm weight has gradient — training will update it",
       "LayerNorm weight has no gradient!")

In [ ]:
# ═══  TEST 9 — Determinism ═══════════════════════════════════════════
#
# WHAT:  Same input → same output, reliably, every time.
# WHY:   Non-determinism makes debugging impossible.
#        In eval mode: must be 100% deterministic (no dropout).
#        In train mode: non-determinism from dropout is EXPECTED.
#
# WE TEST:
#   - eval mode:  two forward passes on same input → identical output
#   - train mode: two forward passes → DIFFERENT (dropout active)
#   - seed fix:   with fixed seed, even train mode is reproducible
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 9: Determinism — Same Input → Same Output?")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
ids = torch.randint(1, VOCAB, (2, 32))

sub("eval() mode — must be perfectly deterministic")
emb.eval()
with torch.no_grad():
    out_a = emb(ids)
    out_b = emb(ids)

identical = torch.allclose(out_a, out_b, atol=0)
result(identical,
       "eval() mode: same input → identical output ✓",
       "eval() mode: outputs DIFFER — critical non-determinism bug!")

sub("train() mode — dropout makes outputs different (intentional)")
emb.train()
with torch.no_grad():
    out_c = emb(ids)
    out_d = emb(ids)

differ = not torch.allclose(out_c, out_d)
dropout_rate = config.DROPOUT
if dropout_rate > 0:
    result(differ,
           f"train() mode: outputs differ due to dropout={dropout_rate} ✓",
           "train() mode: outputs identical — is dropout disabled?")
else:
    result(not differ,
           f"train() mode: outputs identical (dropout=0, expected) ✓",
           "Unexpected behaviour")

sub("Reproducibility with fixed seed")
torch.manual_seed(42)
emb.train()
with torch.no_grad():
    out_seed_a = emb(ids)

torch.manual_seed(42)
with torch.no_grad():
    out_seed_b = emb(ids)

result(torch.allclose(out_seed_a, out_seed_b),
       "Fixed seed → identical outputs in train() mode ✓",
       "Fixed seed still gives different outputs — non-determinism outside dropout!")

In [ ]:
# ═══  TEST 10 — Edge Cases ═══════════════════════════════════════════
#
# WHAT:  Unusual but valid inputs that might appear in production.
# WHY:   Edge cases are where bugs hide. Test them before deployment.
#
# CASES:
#   - Sequence length = 1 (single token — e.g. classification with [CLS] only)
#   - All padding tokens (should give zero after embedding, not after norm)
#   - Batch size = 1 (inference often runs single samples)
#   - Repeated identical sequences in a batch
#   - Very long sequence (at the limit of the RoPE table)
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 10: Edge Cases")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
emb.eval()

edge_cases = [
    (torch.randint(1, VOCAB, (1, 1)),           "batch=1, seq=1 (single token)"),
    (torch.randint(1, VOCAB, (1, MLEN)),        "batch=1, seq=MAX (full sequence)"),
    (torch.randint(1, VOCAB, (32, 1)),          "batch=32, seq=1 (parallel single tokens)"),
    (torch.zeros(2, 16, dtype=torch.long),      "all padding (id=0)"),
    (torch.ones(2, 16, dtype=torch.long),       "all same token (id=1)"),
    (torch.arange(MLEN).unsqueeze(0) % VOCAB,  "sequential IDs 0,1,2,..."),
]

for ids, label in edge_cases:
    try:
        with torch.no_grad():
            out = emb(ids)
        nan_free = not torch.isnan(out).any().item()
        inf_free = not torch.isinf(out).any().item()
        ok_flag  = nan_free and inf_free
        status   = "✅" if ok_flag else "❌"
        issues   = (["NaN"] if not nan_free else []) + (["Inf"] if not inf_free else [])
        print(f"  {status}  {label}")
        print(f"       input={tuple(ids.shape)} → output={tuple(out.shape)}"
              + (f"  ISSUES: {issues}" if issues else ""))
    except Exception as e:
        print(f"  ❌  {label}")
        print(f"       {type(e).__name__}: {e}")

# ── All-padding special case explanation ──────────────────────────────────────
print("""
  NOTE on all-padding:
    Raw embedding of token 0 = zero vector (padding_idx=0).
    After LayerNorm: LayerNorm of a zero vector is undefined
    (std=0 → division by zero, but PyTorch adds epsilon to prevent crash).
    Output may be non-zero and that is CORRECT and SAFE.
    Padding tokens are masked out in the attention layer (Module 3).
""")

In [ ]:
# ═══  TEST 11 — Batch Independence ═══════════════════════════════════
#
# WHAT:  Processing a sample alone vs in a batch must give identical results.
# WHY:   BatchNorm breaks this property (which is why transformers use LayerNorm).
#        If this test fails, your model behaves differently at inference
#        (batch=1) vs training (batch=N) — a very subtle, hard-to-find bug.
#
# HOW:
#   1. Run one sample alone → out_single
#   2. Run same sample in a batch of 4 → out_in_batch
#   3. They must be identical
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 11: Batch Independence — Single vs Batched Inference")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
emb.eval()   # no dropout

# A specific sequence we care about
seq_of_interest = torch.randint(1, VOCAB, (1, 32))  # [1, 32]

# Run it alone
with torch.no_grad():
    out_alone = emb(seq_of_interest)     # [1, 32, DIM]

# Run it as the first item in a batch of 4 different sequences
other_seqs  = torch.randint(1, VOCAB, (3, 32))
batch        = torch.cat([seq_of_interest, other_seqs], dim=0)  # [4, 32]
with torch.no_grad():
    out_batch   = emb(batch)             # [4, 32, DIM]
    out_in_batch = out_batch[0:1]        # [1, 32, DIM]  — our sequence only

identical = torch.allclose(out_alone, out_in_batch, atol=1e-6)
result(identical,
       "Single result == Batched result (LayerNorm is per-sample) ✓",
       "Results DIFFER between single and batched — possible BatchNorm bug!")

print(f"\n  Max absolute difference: {(out_alone - out_in_batch).abs().max().item():.2e}")
print(f"  (Should be < 1e-6 — pure floating-point rounding)")

In [ ]:
# ═══  TEST 12 — cos²+sin²=1 Mathematical Identity ════════════════════
#
# WHAT:  Verify the RoPE tables are mathematically correct.
# WHY:   If the tables are wrong, every forward pass produces garbage.
#        This is the simplest possible sanity check (Pythagorean identity).
#
# ALSO:
#   - Verify position 0 has all-zero sin (no rotation at position 0)
#   - Verify position 0 has all-one  cos (identity rotation)
#   - Verify frequency range (highest freq ≈ 1.0, lowest ≈ 0.001)
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 12: RoPE Table Mathematical Correctness")

rope = RotaryEmbedding(dim=DIM, max_len=MLEN)

sub("Pythagorean identity: cos²(θ) + sin²(θ) = 1 for all entries")
identity = rope.cos_table ** 2 + rope.sin_table ** 2
all_ones = torch.allclose(identity, torch.ones_like(identity), atol=1e-6)
result(all_ones,
       "cos²+sin²=1 holds everywhere in the table ✓",
       f"Pythagorean identity violated! Max deviation: {(identity-1).abs().max().item():.2e}")

sub("Position 0: sin=0, cos=1 (no rotation at first position)")
sin_pos0 = rope.sin_table[0]   # should be all zeros
cos_pos0 = rope.cos_table[0]   # should be all ones
result(torch.allclose(sin_pos0, torch.zeros_like(sin_pos0), atol=1e-7),
       "sin at position 0 = 0 everywhere ✓",
       f"sin at pos 0 not zero! Max: {sin_pos0.abs().max().item():.2e}")
result(torch.allclose(cos_pos0, torch.ones_like(cos_pos0), atol=1e-7),
       "cos at position 0 = 1 everywhere ✓",
       f"cos at pos 0 not one! Min: {cos_pos0.min().item():.4f}")

sub("Frequency range check")
half_dim = DIM // 2
# dim pair 0: θ = 1/(10000^0) = 1.0  → highest frequency
# dim pair -1: θ = 1/(10000^((half_dim-1)/half_dim)) → lowest frequency
theta_0    = 1.0 / (10000.0 ** (0.0 / half_dim))
theta_last = 1.0 / (10000.0 ** ((half_dim - 1) / half_dim))

# Check position 1 (angle = 1 × theta)
sin_pos1 = rope.sin_table[1]
expected_sin_0    = math.sin(theta_0)
expected_sin_last = math.sin(theta_last)

diff_0    = abs(sin_pos1[0].item() - expected_sin_0)
diff_last = abs(sin_pos1[-1].item() - expected_sin_last)

result(diff_0 < 1e-5,
       f"Dim 0 frequency correct: sin(pos=1)={sin_pos1[0].item():.6f} ≈ {expected_sin_0:.6f}",
       f"Dim 0 frequency wrong! diff={diff_0:.2e}")
result(diff_last < 1e-5,
       f"Last dim frequency correct: sin(pos=1)={sin_pos1[-1].item():.6f} ≈ {expected_sin_last:.6f}",
       f"Last dim frequency wrong! diff={diff_last:.2e}")

print(f"\n  θ range: [{theta_last:.6f}, {theta_0:.4f}]")
print(f"  Fast dims rotate by {math.degrees(theta_0):.1f}° per position")
print(f"  Slow dims rotate by {math.degrees(theta_last):.4f}° per position")

In [ ]:
# ═══  TEST 13 — Parameter Count & Memory ═════════════════════════════
#
# WHAT:  Count trainable parameters and estimate memory usage.
# WHY:   In production you need to know if the model fits in GPU memory.
#        Embedding tables are often the LARGEST part of a small model.
#
# EXPECTED (with our config):
#   nn.Embedding      : 32000 × 256 = 8,192,000 params = 32 MB (float32)
#   LayerNorm weight  :      256    params
#   LayerNorm bias    :      256    params
#   RoPE              :        0    trainable params (buffers only)
#   Total             : ~8,192,512 params ≈ 32 MB
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 13: Parameter Count and Memory Estimate")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)

print(f"\n  {'Module':<35}  {'Params':>12}  {'Size (MB)':>10}  {'Trainable'}")
print(f"  {'─'*35}  {'─'*12}  {'─'*10}  {'─'*9}")

total_params     = 0
trainable_params = 0

for name, module in emb.named_modules():
    if not list(module.children()):   # leaf modules only
        n = sum(p.numel() for p in module.parameters())
        t = sum(p.numel() for p in module.parameters() if p.requires_grad)
        b = sum(buf.numel() for buf in module.buffers())
        if n > 0 or b > 0:
            mb = (n * 4) / (1024 ** 2)
            print(f"  {name:<35}  {n:>12,}  {mb:>9.2f}  {'Yes' if t > 0 else 'No (buffer)'}")
        total_params     += n
        trainable_params += t

total_mb = (total_params * 4) / (1024 ** 2)
print(f"\n  {'TOTAL':<35}  {total_params:>12,}  {total_mb:>9.2f}  {trainable_params:,} trainable")

# RoPE buffers (not params)
rope_buf_size = sum(b.numel() for b in emb.rope.buffers()) * 4 / (1024 ** 2)
print(f"\n  RoPE table (buffer, not trained): {rope_buf_size:.3f} MB")

# Expected embedding size check
expected_emb_params = VOCAB * DIM
actual_emb_params   = emb.embedding.weight.numel()
result(actual_emb_params == expected_emb_params,
       f"Embedding table has {actual_emb_params:,} params = {VOCAB:,} × {DIM} ✓",
       f"Embedding table size wrong: {actual_emb_params} ≠ {expected_emb_params}")

In [ ]:
# ═══ TEST 13 — Parameter Count & Memory ═════════════════════════════
#
# WHAT:  Count trainable parameters and estimate memory usage.
# WHY:   In production you need to know if the model fits in GPU memory.
#        Embedding tables are often the LARGEST part of a small model.
#
# EXPECTED (with our config):
#   nn.Embedding      : 32000 × 256 = 8,192,000 params = 32 MB (float32)
#   LayerNorm weight  :      256    params
#   LayerNorm bias    :      256    params
#   RoPE              :        0    trainable params (buffers only)
#   Total             : ~8,192,512 params ≈ 32 MB
# ═════════════════════════════════════════════════════════════════════════════

section("TEST 13: Parameter Count and Memory Estimate")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)

print(f"\n  {'Module':<35}  {'Params':>12}  {'Size (MB)':>10}  {'Trainable'}")
print(f"  {'─'*35}  {'─'*12}  {'─'*10}  {'─'*9}")

total_params     = 0
trainable_params = 0

for name, module in emb.named_modules():
    if not list(module.children()):   # leaf modules only
        n = sum(p.numel() for p in module.parameters())
        t = sum(p.numel() for p in module.parameters() if p.requires_grad)
        b = sum(buf.numel() for buf in module.buffers())
        if n > 0 or b > 0:
            mb = (n * 4) / (1024 ** 2)
            print(f"  {name:<35}  {n:>12,}  {mb:>9.2f}  {'Yes' if t > 0 else 'No (buffer)'}")
        total_params     += n
        trainable_params += t

total_mb = (total_params * 4) / (1024 ** 2)
print(f"\n  {'TOTAL':<35}  {total_params:>12,}  {total_mb:>9.2f}  {trainable_params:,} trainable")

# RoPE buffers (not params)
rope_buf_size = sum(b.numel() for b in emb.rope.buffers()) * 4 / (1024 ** 2)
print(f"\n  RoPE table (buffer, not trained): {rope_buf_size:.3f} MB")

# Expected embedding size check
expected_emb_params = VOCAB * DIM
actual_emb_params   = emb.embedding.weight.numel()
result(actual_emb_params == expected_emb_params,
       f"Embedding table has {actual_emb_params:,} params = {VOCAB:,} × {DIM} ✓",
       f"Embedding table size wrong: {actual_emb_params} ≠ {expected_emb_params}")

In [ ]:
# ═══  VISUALIZATION 1 — RoPE Rotation Pattern ════════════════════════
#
# WHAT:  Plot how RoPE rotates the same vector at different positions.
# WHY:   Numbers alone are hard to understand. Seeing circular rotation
#        makes the intuition click immediately.
#
# PANELS:
#   Top-left  : 2D scatter of (dim0, dim1) across positions — shows rotation
#   Top-right : Each dimension's value across positions — shows oscillation
#   Bottom-left : Heatmap of cos_table — shows frequency structure
#   Bottom-right: Distance from position 0 as position increases
# ═════════════════════════════════════════════════════════════════════════════

section("VISUALIZATION 1: RoPE Rotation Pattern")

rope = RotaryEmbedding(dim=DIM, max_len=MLEN)

N_POS = 100   # number of positions to visualize

# Same base vector repeated at N_POS positions
base  = torch.randn(1, 1, DIM)
seq   = base.expand(1, N_POS, DIM).clone()   # [1, N_POS, DIM]
rot   = rope(seq).detach()                    # [1, N_POS, DIM]

fig = plt.figure(figsize=(16, 12))
fig.suptitle("RoPE: How Position Rotates Token Vectors", fontsize=14, fontweight="bold")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: 2D rotation trace (dim 0 vs dim 1) ───────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
x0  = rot[0, :, 0].numpy()
x1  = rot[0, :, 1].numpy()
sc  = ax1.scatter(x0, x1, c=range(N_POS), cmap="plasma", s=20, zorder=3)
ax1.plot(x0, x1, "k-", alpha=0.2, linewidth=0.8)
plt.colorbar(sc, ax=ax1, label="Position")
ax1.set_title("Dims 0,1: Circular Rotation\n(each dot = one position)")
ax1.set_xlabel("Dimension 0")
ax1.set_ylabel("Dimension 1")
ax1.set_aspect("equal")
# Annotate a few key positions
for pos in [0, 10, 25, 50, 99]:
    ax1.annotate(f"pos {pos}", (x0[pos], x1[pos]), fontsize=7,
                 xytext=(5, 5), textcoords="offset points")

# ── Panel 2: Value of each dim across positions ────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
dim_labels = {0: "Dim 0 (fast, θ=1.0)", 2: "Dim 2", 64: "Dim 64 (medium)", 254: "Dim 254 (slow)"}
colors     = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
for (d, lbl), col in zip(dim_labels.items(), colors):
    ax2.plot(rot[0, :, d].numpy(), label=lbl, color=col, linewidth=1.2)
ax2.set_title("Dimension Values Across Positions\n(fast dims oscillate quickly)")
ax2.set_xlabel("Position")
ax2.set_ylabel("Value")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# ── Panel 3: cos_table heatmap ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
# Show first 64 positions and first 32 frequency pairs for clarity
cos_vis = rope.cos_table[:64, :32].numpy()
im      = ax3.imshow(cos_vis, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax3, label="cos(pos × θ)")
ax3.set_title("cos_table[:64, :32]\n(rows=positions, cols=frequency pairs)")
ax3.set_xlabel("Frequency pair index (0=fast, 31=slow)")
ax3.set_ylabel("Position")
ax3.set_xticks([0, 8, 16, 24, 31])
ax3.set_yticks([0, 16, 32, 48, 63])

# ── Panel 4: Distance from position 0 ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
dist_from_0 = [(rot[0, p] - rot[0, 0]).norm().item() for p in range(N_POS)]
ax4.plot(dist_from_0, "tab:purple", linewidth=1.5)
ax4.fill_between(range(N_POS), dist_from_0, alpha=0.2, color="tab:purple")
ax4.set_title("Distance from Position 0\n(grows then oscillates — expected for RoPE)")
ax4.set_xlabel("Position")
ax4.set_ylabel("|rotated_vec_pos_p  −  rotated_vec_pos_0|")
ax4.grid(alpha=0.3)

save_path = os.path.join(RESULTS, "rope_rotation_pattern.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n  Saved → {save_path}")

In [ ]:
# ═══  VISUALIZATION 2 — Embedding Space ══════════════════════════════
#
# WHAT:  Show that similar token IDs do NOT necessarily have similar vectors
#        (embeddings are learned, not ordered by ID).
#        Also show that LayerNorm makes all vectors unit-scale.
#
# PANELS:
#   Left  : Histogram of embedding norms before and after LayerNorm
#   Middle: Histogram of embedding values (should be roughly Gaussian)
#   Right : Cosine similarity matrix for 10 random tokens
# ═════════════════════════════════════════════════════════════════════════════

section("VISUALIZATION 2: Embedding Space Properties")

emb = TokenEmbedding(vocab_size=VOCAB, embed_dim=DIM, max_len=MLEN)
emb.eval()

# Sample 200 random token IDs
n_tokens  = 200
token_ids = torch.randint(1, VOCAB, (1, n_tokens))

with torch.no_grad():
    raw_emb  = emb.embedding(token_ids)[0]   # [n_tokens, DIM] — raw lookup only
    full_out = emb(token_ids)[0]             # [n_tokens, DIM] — full pipeline

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Embedding Space Properties", fontsize=13, fontweight="bold")

# ── Panel 1: Norm distribution before and after LayerNorm ─────────────────────
ax = axes[0]
raw_norms  = raw_emb.norm(dim=-1).numpy()
full_norms = full_out.norm(dim=-1).numpy()

ax.hist(raw_norms,  bins=30, alpha=0.6, label=f"Raw embedding\nμ={raw_norms.mean():.1f}",
        color="tab:blue")
ax.hist(full_norms, bins=30, alpha=0.6, label=f"After LayerNorm\nμ={full_norms.mean():.1f}",
        color="tab:orange")
ax.set_title("Norm Distribution\n(LayerNorm stabilises norms)")
ax.set_xlabel("L2 Norm of vector")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Panel 2: Value histogram ───────────────────────────────────────────────────
ax = axes[1]
ax.hist(raw_emb.flatten().numpy(),  bins=60, alpha=0.5, label="Raw embedding",
        color="tab:blue", density=True)
ax.hist(full_out.flatten().numpy(), bins=60, alpha=0.5, label="After LayerNorm",
        color="tab:orange", density=True)
ax.set_title("Value Distribution\n(should be roughly Gaussian)")
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# ── Panel 3: Cosine similarity matrix for 10 tokens ───────────────────────────
ax = axes[2]
n_show     = 10
ids_show   = torch.randint(1, VOCAB, (1, n_show))
with torch.no_grad():
    vecs = emb(ids_show)[0]                      # [n_show, DIM]

# Normalise for cosine similarity
vecs_n = vecs / vecs.norm(dim=-1, keepdim=True)
sim    = (vecs_n @ vecs_n.T).numpy()

im = ax.imshow(sim, cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label="Cosine similarity")
ax.set_title("Cosine Similarity Matrix\n(random tokens — mostly uncorrelated)")
ax.set_xlabel("Token index")
ax.set_ylabel("Token index")
ax.set_xticks(range(n_show))
ax.set_yticks(range(n_show))
ax.set_xticklabels([f"T{ids_show[0,i].item()}" for i in range(n_show)],
                    rotation=45, ha="right", fontsize=7)
ax.set_yticklabels([f"T{ids_show[0,i].item()}" for i in range(n_show)], fontsize=7)

plt.tight_layout()
save_path = os.path.join(RESULTS, "embedding_space.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n  Saved → {save_path}")

In [ ]:
# ═══  VISUALIZATION 3 — "I love Cats" Full Walkthrough ═══════════════
#
# WHAT:  Trace the EXACT matrices for "I love Cats" through the full pipeline.
#        Uses dim=8 so all numbers are printable and understandable.
# WHY:   Seeing real numbers makes the math concrete.
#        Every step matches the formulas in the RoPE paper.
# ═════════════════════════════════════════════════════════════════════════════

section("VISUALIZATION 3: 'I love Cats' Full Matrix Walkthrough (dim=8)")

torch.manual_seed(0)   # fixed seed for reproducible "embedding" values

MINI_DIM   = 8
MINI_VOCAB = 1000

# Build a tiny model with dim=8 so we can print all values
rope8 = RotaryEmbedding(dim=MINI_DIM, max_len=16)

# Simulated token IDs: "I"=27, "love"=342, "Cats"=891
# Use a fixed seed so the random embedding table is reproducible
torch.manual_seed(0)
emb8  = nn.Embedding(MINI_VOCAB, MINI_DIM, padding_idx=0)
token_ids_demo = torch.tensor([[27, 342, 891]])   # [1, 3]

with torch.no_grad():
    raw = emb8(token_ids_demo)   # [1, 3, 8]

words = ["I   ", "love", "Cats"]

def fmt(t):
    """Format a 1D tensor as a row of fixed-width numbers."""
    return "  ".join(f"{v:+.4f}" for v in t.tolist())

print(f"\n{'═'*65}")
print(f"  SENTENCE: 'I love Cats'")
print(f"  Token IDs: [27, 342, 891]   dim={MINI_DIM}   seq_len=3")
print(f"{'═'*65}")

print(f"\n  ── STEP 1: nn.Embedding Lookup ──────────────────────────")
print(f"  {'Word':<6}  {'d0':>8}  {'d1':>8}  {'d2':>8}  {'d3':>8}  "
      f"{'d4':>8}  {'d5':>8}  {'d6':>8}  {'d7':>8}")
print(f"  {'─'*78}")
for i, word in enumerate(words):
    print(f"  {word}   {fmt(raw[0, i])}")

print(f"\n  ── STEP 2: RoPE Frequency Table (dim=8, half_dim=4) ────")
half = MINI_DIM // 2
i_vals = torch.arange(half).float()
theta  = 1.0 / (10000.0 ** (i_vals / half))
print(f"  i            :  {i_vals.tolist()}")
print(f"  2i/dim       :  {(i_vals/half).tolist()}")
print(f"  10000^(2i/d) :  {[(10000**(iv.item()/half)) for iv in i_vals]}")
print(f"  theta (1/^)  :  {theta.tolist()}")

positions = torch.arange(3).float()
angles    = torch.outer(positions, theta)   # [3, 4]

print(f"\n  Angle matrix  (positions × theta):")
print(f"  {'':10}  {'θ_0':>10}  {'θ_1':>10}  {'θ_2':>10}  {'θ_3':>10}")
for p in range(3):
    ang_str = "  ".join(f"{v:+.6f}" for v in angles[p].tolist())
    print(f"  pos {p} ({words[p]}): {ang_str}")

cos_t = torch.cos(angles)   # [3, 4]
sin_t = torch.sin(angles)   # [3, 4]
print(f"\n  cos table:")
for p in range(3):
    print(f"  pos {p}: {fmt(cos_t[p])}")
print(f"\n  sin table:")
for p in range(3):
    print(f"  pos {p}: {fmt(sin_t[p])}")

print(f"\n  ── STEP 3: Split into even/odd dims ────────────────────")
x_even = raw[0, :, 0::2]   # [3, 4]  dims 0,2,4,6
x_odd  = raw[0, :, 1::2]   # [3, 4]  dims 1,3,5,7
print(f"  {'Word':<6}  {'even (d0,d2,d4,d6)':<44}  {'odd (d1,d3,d5,d7)'}")
for i, word in enumerate(words):
    print(f"  {word}   [{fmt(x_even[i])}]   [{fmt(x_odd[i])}]")

print(f"\n  ── STEP 4: Apply Rotation ──────────────────────────────")
print(f"  Formula: new_even = even*cos - odd*sin")
print(f"           new_odd  = even*sin + odd*cos")
new_even = x_even * cos_t - x_odd * sin_t
new_odd  = x_even * sin_t + x_odd * cos_t

for i, word in enumerate(words):
    print(f"\n  {word}  (pos {i}):")
    print(f"    cos         = [{fmt(cos_t[i])}]")
    print(f"    sin         = [{fmt(sin_t[i])}]")
    print(f"    new_even    = [{fmt(new_even[i])}]")
    print(f"    new_odd     = [{fmt(new_odd[i])}]")

print(f"\n  ── STEP 5: Interleave back ─────────────────────────────")
stacked   = torch.stack([new_even, new_odd], dim=-1)   # [3, 4, 2]
x_rotated = stacked.flatten(start_dim=-2)               # [3, 8]
print(f"  {'Word':<6}  Final rotated vector (all 8 dims)")
for i, word in enumerate(words):
    print(f"  {word}   [{fmt(x_rotated[i])}]")

print(f"\n  ── Verify: position 0 vector unchanged? ────────────────")
unchanged = torch.allclose(raw[0, 0], x_rotated[0], atol=1e-6)
result(unchanged,
       "'I' at pos=0 unchanged (rotation by 0° = identity)",
       "'I' at pos=0 was changed — bug!")

In [ ]:
# ═══ FINAL SUMMARY ══════════════════════════════════════════════════
#
# Print a concise table of all test results.
# In production CI you would collect these into a dict and assert all pass.
# ═════════════════════════════════════════════════════════════════════════════

section("FINAL SUMMARY")

tests = [
    ("TEST 1",  "Output shape [batch, seq, dim]"),
    ("TEST 2",  "RoPE position sensitivity + monotone distance"),
    ("TEST 3",  "RoPE shape preservation (incl. long sequences)"),
    ("TEST 4",  "Numerical stability (NaN/Inf/extreme values)"),
    ("TEST 5",  "Wrong dtype handling"),
    ("TEST 6",  "Out-of-vocabulary ID handling"),
    ("TEST 7",  "Device mismatch detection"),
    ("TEST 8",  "Gradient flow (model can learn)"),
    ("TEST 9",  "Determinism (eval=deterministic, train=stochastic)"),
    ("TEST 10", "Edge cases (seq=1, all-pad, sequential IDs)"),
    ("TEST 11", "Batch independence (single == batched)"),
    ("TEST 12", "RoPE math identity (cos²+sin²=1)"),
    ("TEST 13", "Parameter count and memory"),
    ("VIZ 1",   "RoPE rotation heatmaps saved"),
    ("VIZ 2",   "Embedding space plots saved"),
    ("VIZ 3",   "'I love Cats' full matrix walkthrough"),
]

print(f"\n  {'ID':<10}  {'Description':<50}  Note")
print(f"  {'─'*10}  {'─'*50}  {'─'*20}")
for tid, desc in tests:
    print(f"  {tid:<10}  {desc:<50}  → see cell output above")

print(f"""
  PRODUCTION CHECKLIST:
  ─────────────────────────────────────────────────────────
  ✅  Always call emb.eval() before inference
  ✅  Always cast ids to int64: ids = ids.long()
  ✅  Always move ids to model device: ids = ids.to(device)
  ✅  Clamp IDs: ids = ids.clamp(0, vocab_size - 1)
  ✅  Use torch.no_grad() during inference (saves memory)
  ✅  RoPE table covers 4× MAX_SEQ_LEN (generalizes safely)
  ✅  padding_idx=0 in nn.Embedding (no gradient for <pad>)

  FILES SAVED:
  ─────────────────────────────────────────────────────────
  results/rope_rotation_pattern.png
  results/embedding_space.png
""")
print("  Module 2 fully verified — ready to build Module 3: Attention")